# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# 2D Image Fundamentals and Geometric Transformations
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/02_Transforms_ImageFundamentals.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
This interactive Jupyter Notebook introduces the foundational mathematical models, data representations, spatial characteristics, and geometric transformations of **2D Digital Images**.

By completing this notebook, you will learn to:
1. Understand **2D spatial matrices**, intensity quantization, data types (`uint8`, `int32`, `float32`, `float64`), and programmatically measure memory footprint.
2. Analyze **spatial sampling**, resolution reduction, sub-sampling degradation, and evaluate interpolation techniques (**Nearest-Neighbor, Bilinear, Bicubic, Lanczos**) using **residual difference maps**.
3. Identify **intensity extremas** (global minimum and maximum), compute pixel coordinates, and evaluate spatial distance metrics (**Euclidean $L_2$, Manhattan $L_1$, Chebyshev $L_\infty$**).
4. Perform **bit-plane decomposition** of 8-bit images, evaluate structural contributions from Most Significant Bit (MSB) to Least Significant Bit (LSB), and measure LSB information content.
5. Derive and execute 2D **Affine Transformations** using $3 \times 3$ homogeneous coordinate matrices (Translation, Rotation, Flipping, Shearing, Composite) alongside **Non-Linear Deformable Warping**.

---


---
## SECTION 0: Google Colab Environment Setup & Asset Check

### Theoretical & Engineering Setup
To ensure seamless cross-platform execution (whether locally or inside Google Colab), this environment setup cell automatically initializes required directory trees (`./lab_materials/`), downloads standard test image assets if missing, and sets up high-resolution Matplotlib styling defaults.


In [ ]:
# Section 0: Google Colab Environment Setup & Asset Check
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage
from PIL import Image

# 1. Ensure target directory structure exists
lab_dir = "./example_data"
os.makedirs(lab_dir, exist_ok=True)

# 2. Automated download of required image assets if missing
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Astarakee/isa-su/main/labs/example_data"
ASSETS = {
    "cameraman.tif": f"{GITHUB_RAW_BASE}/cameraman.tif",
    "lena.jpg": f"{GITHUB_RAW_BASE}/lena.jpg",
    "baboon.jpg": f"{GITHUB_RAW_BASE}/baboon.jpg",
    "mri_t1n_brain.png": f"{GITHUB_RAW_BASE}/mri_t1n_brain.png"
}

for fname, url in ASSETS.items():
    fpath = os.path.join(lab_dir, fname)
    if not os.path.exists(fpath):
        print(f"Downloading missing asset '{fname}' into {lab_dir}...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp, open(fpath, 'wb') as f:
                f.write(resp.read())
            print(f"Successfully downloaded '{fname}'")
        except Exception as e:
            print(f"Warning: Failed to download '{fname}': {e}")

# Verify clean image format for cameraman if downloaded as TIFF
cameraman_path = os.path.join(lab_dir, "cameraman.tif")
if os.path.exists(cameraman_path):
    try:
        pil_cam = Image.open(cameraman_path).convert('L')
        pil_cam.save(cameraman_path)
    except Exception as e:
        print(f"TIFF cleanup notice: {e}")

# 3. Configure high-quality inline plotting defaults
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = False
plt.rcParams['figure.autolayout'] = True

print("Environment setup complete. All required libraries and assets are ready.")


---
## SECTION 1: Synthetic Images, Data Types, and Memory Footprint

### Theoretical Background
A continuous 2D spatial image $f(x, y)$ is discretized into a 2D digital matrix $I[m, n]$, where $m \in \{0, 1, \dots, M-1\}$ represents the row index (vertical coordinate) and $n \in \{0, 1, \dots, N-1\}$ represents the column index (horizontal coordinate).

#### Numerical Data Types & Bit Depth
The intensity value at each pixel $I[m, n]$ is stored using specific numerical representations:
- **Unsigned 8-bit Integer (`uint8`)**: Standard for digital display. Range $[0, 255]$ ($2^8 = 256$ discrete levels). Uses 1 byte (8 bits) per pixel.
- **Signed 32-bit Integer (`int32`)**: Used for high-dynamic range indices or discrete intermediate operations. Range $[-2^{31}, 2^{31}-1]$. Uses 4 bytes (32 bits) per pixel.
- **Single-Precision Float (`float32`)**: Standard in machine learning and continuous image filtering. Normalized range typically $[0.0, 1.0]$. Uses 4 bytes (32 bits) per pixel.
- **Double-Precision Float (`float64`)**: High-precision scientific computations. Uses 8 bytes (64 bits) per pixel.

#### Memory Footprint Formulation
For an $M \times N$ single-channel grayscale image:
$$\text{Memory (bits)} = M \times N \times b_p$$
$$\text{Memory (bytes)} = \frac{\text{Memory (bits)}}{8} = M \times N \times S_{\text{dtype}}$$
where $b_p$ is the bit depth (bits per pixel) and $S_{\text{dtype}}$ is the size in bytes per element.


In [ ]:
# Section 1: Code Implementation

# 1. Synthesize Uniform Random Noise Matrix (200 x 200)
np.random.seed(42)
grid_h, grid_w = 200, 200
rand_img = np.random.uniform(low=0.0, high=255.0, size=(grid_h, grid_w))

# 2. Synthesize Structured Chessboard Pattern (200 x 200)
block_size = 25
x_grid, y_grid = np.meshgrid(np.arange(grid_w), np.arange(grid_h))
chessboard_img = (((x_grid // block_size) + (y_grid // block_size)) % 2) * 255.0

# Visualize Synthetic Images
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

im0 = axes[0].imshow(rand_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Uniform Random Noise (200x200)")
axes[0].set_xlabel("Column Index (n)")
axes[0].set_ylabel("Row Index (m)")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(chessboard_img, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("Structured Chessboard Pattern (200x200)")
axes[1].set_xlabel("Column Index (n)")
axes[1].set_ylabel("Row Index (m)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 3. Data Type Conversions & Memory Footprint Computation
dtypes = [np.uint8, np.int32, np.float32, np.float64]

print("=" * 75)
print(f"{'Data Type':<12} | {'Bit Depth':<10} | {'Bytes / Pixel':<14} | {'Total Bytes':<12} | {'Total Bits':<12}")
print("=" * 75)

for dt in dtypes:
    converted_arr = chessboard_img.astype(dt)
    bytes_per_elem = converted_arr.itemsize
    bits_per_elem = bytes_per_elem * 8
    total_bytes = converted_arr.nbytes
    total_bits = total_bytes * 8
    dt_name = dt.__name__
    print(f"{dt_name:<12} | {bits_per_elem:<10} | {bytes_per_elem:<14} | {total_bytes:<12} | {total_bits:<12}")

print("=" * 75)


---
## SECTION 2: Spatial Sampling, Resolution, Interpolation, and Residual Analysis

### Theoretical Background
Spatial resolution refers to the density of pixels representing real-world physical structures. Sub-sampling (downsampling) reduces spatial resolution by discarding pixel samples:

#### Sub-Sampling & Aliasing
Downsampling an image by a factor $S$ reduces dimensions from $M \times N$ to $\lfloor M/S \rfloor \times \lfloor N/S \rfloor$. According to the 2D Nyquist-Shannon Sampling Theorem, downsampling high-frequency content below twice its maximum frequency introduces **aliasing artifacts**, blurriness, and irreversible edge loss.

#### Image Interpolation Principles
When upsampling a low-resolution image back to higher dimensions, intermediate pixel intensities are estimated using surrounding spatial neighbors:
1. **Nearest-Neighbor Interpolation**: Takes the intensity of the closest integer pixel coordinate. Fast, but produces blocky step artifacts.
   $$I(x, y) = I(\lfloor x + 0.5 \rfloor, \lfloor y + 0.5 \rfloor)$$
2. **Bilinear Interpolation**: Weighted linear average of the $2 \times 2$ pixel neighborhood. Smooths transitions but attenuates sharp high frequencies.
   $$I(x,y) = (1-\Delta x)(1-\Delta y)I_0 + \Delta x(1-\Delta y)I_1 + (1-\Delta x)\Delta y I_2 + \Delta x \Delta y I_3$$
3. **Bicubic Interpolation**: Weighted cubic convolution over a $4 \times 4$ pixel neighborhood using cubic splines. Preserves sharp boundaries and derivative continuity.
4. **Lanczos Interpolation**: Uses a windowed $\text{sinc}$ kernel $L(x) = \text{sinc}(x)\text{sinc}(x/a)$ over an $a \times a$ neighborhood. Optimal band-limited frequency reconstruction.

#### Residual / Difference Map Definition
To quantify reconstruction errors resulting from downsampling and subsequent upsampling, we calculate absolute spatial residual maps:
$$R(x, y) = \big| I_{\text{orig}}(x, y) - I_{\text{upsampled}}(x, y) \big|$$
High intensity values in $R(x, y)$ highlight regions where high spatial frequencies (edges, contours) were destroyed.


In [ ]:
# Section 2: Code Implementation

# 1. Load Cameraman Image and Analyze Statistics
cam_path = os.path.join(lab_dir, "cameraman.tif")
img_cam_raw = cv2.imread(cam_path, cv2.IMREAD_GRAYSCALE)

# Handle fallback if missing or unreadable
if img_cam_raw is None:
    print("Warning: cameraman.tif not found or unreadable. Generating synthetic high-contrast test target.")
    img_cam_raw = (np.outer(np.sin(np.linspace(0, 10, 256)), np.cos(np.linspace(0, 10, 256))) * 127.5 + 127.5).astype(np.uint8)

# Print Statistics
print(f"Cameraman Image Characteristics:")
print(f" - Dimensions: {img_cam_raw.shape[0]} x {img_cam_raw.shape[1]} pixels")
print(f" - Data Type:  {img_cam_raw.dtype}")
print(f" - Min Value:  {np.min(img_cam_raw)}")
print(f" - Max Value:  {np.max(img_cam_raw)}")
print(f" - Mean Value: {np.mean(img_cam_raw):.2f}")
print(f" - Std Dev:    {np.std(img_cam_raw):.2f}")

# Display Original Image
plt.figure(figsize=(4.5, 4.5))
plt.imshow(img_cam_raw, cmap='gray')
plt.title(f"Original Image ({img_cam_raw.shape[1]}x{img_cam_raw.shape[0]})")
plt.axis('off')
plt.show()

# 2. Multi-Factor Downsampling (Factors: 2, 4, 8, 16)
downsample_factors = [2, 4, 8, 16]
downsampled_images = []

orig_h, orig_w = img_cam_raw.shape

fig, axes = plt.subplots(1, len(downsample_factors), figsize=(14, 3.5))
for idx, factor in enumerate(downsample_factors):
    new_w, new_h = orig_w // factor, orig_h // factor
    ds_img = cv2.resize(img_cam_raw, (new_w, new_h), interpolation=cv2.INTER_AREA)
    downsampled_images.append(ds_img)
    print(f"Downsampled Factor 1/{factor:<2} -> New Resolution: {new_w} x {new_h}")
    
    axes[idx].imshow(ds_img, cmap='gray')
    axes[idx].set_title(f"Factor 1/{factor} ({new_w}x{new_h})")
    axes[idx].axis('off')

plt.suptitle("Downsampled Representations", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

# 3. Upsampling Back to Original Dimensions using Bilinear Interpolation
upsampled_images = []
fig, axes = plt.subplots(1, len(downsample_factors), figsize=(14, 3.5))

for idx, (factor, ds_img) in enumerate(zip(downsample_factors, downsampled_images)):
    us_img = cv2.resize(ds_img, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    upsampled_images.append(us_img)
    
    axes[idx].imshow(us_img, cmap='gray')
    axes[idx].set_title(f"Upsampled from 1/{factor}")
    axes[idx].axis('off')

plt.suptitle("Upsampled Images (Bilinear Interpolation to Original Size)", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

# 4. Residual / Difference Map Analysis
residual_maps = []
fig, axes = plt.subplots(1, len(downsample_factors), figsize=(16, 3.8))

for idx, (factor, us_img) in enumerate(zip(downsample_factors, upsampled_images)):
    # Compute absolute residual
    res_map = np.abs(img_cam_raw.astype(np.float32) - us_img.astype(np.float32))
    residual_maps.append(res_map)
    mean_err = np.mean(res_map)
    
    im = axes[idx].imshow(res_map, cmap='hot', vmin=0, vmax=np.max(res_map))
    axes[idx].set_title(f"Residual 1/{factor}\n(MAE: {mean_err:.2f})")
    axes[idx].axis('off')
    fig.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.suptitle("Residual Difference Maps R(x,y) = |I_orig - I_upsampled|", fontsize=12, y=1.05)
plt.tight_layout()
plt.show()


---
## SECTION 3: Intensity Extremas, Pixel Coordinates, and Distance Metrics

### Theoretical Background
In digital image processing, a 2D pixel coordinate is denoted by $P(x, y)$, where $x$ represents the column coordinate (horizontal distance from left margin) and $y$ represents the row coordinate (vertical distance from top margin).

#### Extremas Identification
Global intensity extremas are defined as the spatial grid coordinates corresponding to minimum and maximum intensity values:
$$(x_{\min}, y_{\min}) = \arg\min_{(x,y)} I(x, y), \quad (x_{\max}, y_{\max}) = \arg\max_{(x,y)} I(x, y)$$

#### Distance Metrics in 2D Spatial Grids
Given two spatial points $P_1(x_1, y_1)$ and $P_2(x_2, y_2)$:
1. **Euclidean Distance ($L_2$ norm)**: Measures straight-line physical distance.
   $$D_E(P_1, P_2) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$
2. **Manhattan / City-Block Distance ($L_1$ norm)**: Distance traversed along orthogonal grid axes.
   $$D_M(P_1, P_2) = |x_1 - x_2| + |y_1 - y_2|$$
3. **Chebyshev Distance ($L_\infty$ norm)**: Maximum absolute coordinate offset (chessboard movement distance).
   $$D_C(P_1, P_2) = \max\left(|x_1 - x_2|, |y_1 - y_2|\right)$$


In [ ]:
# Section 3: Code Implementation

# 1. Load Lena Image and Read Intensity Extremas
lena_path = os.path.join(lab_dir, "lena.jpg")
img_lena = cv2.imread(lena_path, cv2.IMREAD_GRAYSCALE)

if img_lena is None:
    print("Warning: lena.jpg not found. Creating synthetic gradient target.")
    img_lena = (np.linspace(0, 255, 256*256).reshape(256, 256)).astype(np.uint8)

min_val = np.min(img_lena)
max_val = np.max(img_lena)

# 2. Programmatically Locate Coordinates (Note: np.unravel_index returns (row, col) = (y, x))
y_min, x_min = np.unravel_index(np.argmin(img_lena), img_lena.shape)
y_max, x_max = np.unravel_index(np.argmax(img_lena), img_lena.shape)

print(f"Lena Image Statistics & Spatial Extremas:")
print(f" - Global Minimum Intensity: {min_val} at Coordinate P_min(x={x_min}, y={y_min})")
print(f" - Global Maximum Intensity: {max_val} at Coordinate P_max(x={x_max}, y={y_max})")

# 3. Visualize Image with Overlay Extremas Markers
plt.figure(figsize=(6, 6))
plt.imshow(img_lena, cmap='gray')
plt.title("Lena Image with Highlighted Intensity Extremas")

# Overlay Markers: Red Circle for Min, Green Star for Max
plt.plot(x_min, y_min, 'ro', markersize=10, markeredgewidth=2, label=f"Global Min ({min_val}) at ({x_min},{y_min})")
plt.plot(x_max, y_max, 'g*', markersize=14, markeredgewidth=2, label=f"Global Max ({max_val}) at ({x_max},{y_max})")

# Annotate Text Labels
plt.annotate(f"Min ({min_val})", (x_min, y_min), xytext=(x_min+15, y_min+15),
             color='red', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", lw=1.5))
plt.annotate(f"Max ({max_val})", (x_max, y_max), xytext=(x_max+15, y_max-15),
             color='green', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="green", lw=1.5))

plt.xlabel("Column Coordinate (x)")
plt.ylabel("Row Coordinate (y)")
plt.legend(loc='lower right')
plt.show()

# 4. Programmatically Compute Distance Metrics between P_min and P_max
dx = abs(x_min - x_max)
dy = abs(y_min - y_max)

dist_euclidean = np.sqrt(dx**2 + dy**2)
dist_manhattan = dx + dy
dist_chebyshev = max(dx, dy)

print("=" * 65)
print(f"Distance Metrics between P_min({x_min}, {y_min}) and P_max({x_max}, {y_max}):")
print("=" * 65)
print(f" - Euclidean Distance (L2 Norm)   : {dist_euclidean:.4f} pixels")
print(f" - Manhattan Distance (L1 Norm)   : {dist_manhattan} pixels")
print(f" - Chebyshev Distance (Linf Norm) : {dist_chebyshev} pixels")
print("=" * 65)


---
## SECTION 4: Bit-Plane Decomposition & Least Significant Bit (LSB) Analysis

### Theoretical Background
An 8-bit digital grayscale pixel intensity $I(x, y) \in [0, 255]$ can be expressed as a linear combination of eight binary bit planes $b_0(x,y), b_1(x,y), \dots, b_7(x,y) \in \{0, 1\}$:

$$I(x, y) = \sum_{k=0}^{7} b_k(x, y) \cdot 2^k = b_7 2^7 + b_6 2^6 + b_5 2^5 + b_4 2^4 + b_3 2^3 + b_2 2^2 + b_1 2^1 + b_0 2^0$$

#### Structural Significance: MSB vs. LSB
- **Most Significant Bit (MSB, Bit 7)**: Carries a weighting factor of $2^7 = 128$ ($50\%$ of the total intensity dynamic range). It encodes primary geometric boundaries and structural contrast.
- **Least Significant Bit (LSB, Bit 0)**: Carries a weighting factor of $2^0 = 1$. It encodes imperceptible fine amplitude variations and high-frequency sensor noise.

#### Steganographic & Information Content
Because zeroing out or modifying the LSB plane causes at most a $\pm 1$ intensity change per pixel ($R(x,y) \le 1$), LSB modification is visually imperceptible to human vision, making it useful for digital watermarking and steganography.


In [ ]:
# Section 4: Code Implementation

# 1. Load Baboon Image
baboon_path = os.path.join(lab_dir, "baboon.jpg")
img_baboon = cv2.imread(baboon_path, cv2.IMREAD_GRAYSCALE)

if img_baboon is None:
    print("Warning: baboon.jpg not found. Synthesizing random texture image.")
    img_baboon = np.random.randint(0, 256, (256, 256), dtype=np.uint8)

# 2. Extract and Display All 8 Bit Planes
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

bit_planes = []
for k in range(8):
    # Extract k-th bit using bitwise right-shift and bitwise AND
    bit_plane = (img_baboon >> k) & 1
    bit_planes.append(bit_plane)
    
    axes[k].imshow(bit_plane, cmap='gray')
    axes[k].set_title(f"Bit Plane {k} (Weight: {2**k})")
    axes[k].axis('off')

plt.suptitle("Complete Bit-Plane Decomposition (Bit 0 = LSB, Bit 7 = MSB)", fontsize=13)
plt.tight_layout()
plt.show()

# 3. LSB Plane Extraction and Zeroing Analysis
lsb_plane = bit_planes[0]

# Zero out the LSB plane (Bit 0) using bitwise AND with mask 0xFE (11111110 binary = 254)
img_lsb_zeroed = img_baboon & 0xFE

# Compute Absolute Difference Map
abs_diff_lsb = np.abs(img_baboon.astype(np.float32) - img_lsb_zeroed.astype(np.float32))

# 4. Display Visual Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

im0 = axes[0].imshow(img_baboon, cmap='gray')
axes[0].set_title("Original Baboon Image")
axes[0].axis('off')

im1 = axes[1].imshow(img_lsb_zeroed, cmap='gray')
axes[1].set_title("LSB-Modified Image (Bit 0 Zeroed Out)")
axes[1].axis('off')

im2 = axes[2].imshow(abs_diff_lsb, cmap='hot', vmin=0, vmax=1)
axes[2].set_title("Absolute Difference Map |I_orig - I_modified|")
axes[2].axis('off')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"LSB Modification Summary:")
print(f" - Maximum absolute error introduced per pixel: {np.max(abs_diff_lsb):.0f} intensity level")
print(f" - Mean absolute error across entire image:     {np.mean(abs_diff_lsb):.4f} intensity level")


---
## SECTION 5: Geometric Transformations: Affine and Deformable

### Theoretical Background
Geometric transformations warp spatial coordinates of an image $I(x, y)$ to new coordinates $I'(x', y')$.

#### 2D Affine Transformations
Affine transformations preserve points, straight lines, and parallelism. Expressed in $3 \times 3$ homogeneous coordinate matrix form:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \mathbf{M} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

1. **Translation (Shifting)**: Shifts origin by $(t_x, t_y)$.
   $$\mathbf{M}_{\text{shift}} = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}$$
2. **Rotation**: Rotates grid by angle $\theta$ around a pivot center $(x_c, y_c)$.
3. **Reflection (Flipping)**: Mirroring along horizontal ($x$-axis) or vertical ($y$-axis).
4. **Shearing**: Skews axes proportionally:
   $$\mathbf{M}_{\text{shear}} = \begin{bmatrix} 1 & s_x & 0 \\ s_y & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$
5. **Composite Transformation**: Matrix multiplication of sequential affine operations $\mathbf{M}_{\text{comp}} = \mathbf{M}_3 \cdot \mathbf{M}_2 \cdot \mathbf{M}_1$.

#### Non-Linear / Deformable Transformations
Non-rigid transformations allow spatially varying displacements $u(x,y)$ and $v(x,y)$:
$$x' = x + u(x, y), \quad y' = y + v(x, y)$$
Common in medical image registration (e.g., elastic brain warping, cardiac motion tracking).


In [ ]:
# Section 5: Code Implementation

# 1. Load Brain MRI Medical Image
mri_path = os.path.join(lab_dir, "mri_t1n_brain.png")
img_mri = cv2.imread(mri_path, cv2.IMREAD_GRAYSCALE)

if img_mri is None:
    print("Warning: mri_t1n_brain.png not found. Generating synthetic brain ellipse model.")
    h_m, w_m = 300, 300
    y_m, x_m = np.ogrid[:h_m, :w_m]
    mask = ((x_m - 150)**2 / 80**2 + (y_m - 150)**2 / 110**2) <= 1
    img_mri = (mask * 200 + np.random.randint(0, 30, (h_m, w_m))).astype(np.uint8)

h, w = img_mri.shape[:2]

# Define 9 Transformations
transformations = []

# 1. Horizontal Shifting (tx = +30, ty = 0)
M_hshift = np.float32([[1, 0, 30], [0, 1, 0]])
t1 = cv2.warpAffine(img_mri, M_hshift, (w, h))
transformations.append(("1. Horizontal Shift (+30px)", t1))

# 2. Vertical Shifting (tx = 0, ty = +40)
M_vshift = np.float32([[1, 0, 0], [0, 1, 40]])
t2 = cv2.warpAffine(img_mri, M_vshift, (w, h))
transformations.append(("2. Vertical Shift (+40px)", t2))

# 3. Combined Shifting (tx = +30, ty = +40)
M_cshift = np.float32([[1, 0, 30], [0, 1, 40]])
t3 = cv2.warpAffine(img_mri, M_cshift, (w, h))
transformations.append(("3. Combined Shift (+30, +40)", t3))

# 4. Clockwise Rotation (10 deg around image center)
center = (w // 2, h // 2)
M_rot = cv2.getRotationMatrix2D(center, -10, 1.0)
t4 = cv2.warpAffine(img_mri, M_rot, (w, h))
transformations.append(("4. Clockwise Rotation (-10 deg)", t4))

# 5. Horizontal Flipping
t5 = cv2.flip(img_mri, 1)
transformations.append(("5. Horizontal Flipping", t5))

# 6. Vertical Flipping
t6 = cv2.flip(img_mri, 0)
transformations.append(("6. Vertical Flipping", t6))

# 7. Shearing Transformation (sx = 0.2, sy = 0)
M_shear = np.float32([[1, 0.2, 0], [0, 1, 0]])
t7 = cv2.warpAffine(img_mri, M_shear, (w, h))
transformations.append(("7. Horizontal Shearing (sx=0.2)", t7))

# 8. Composite Affine Transformation (Shift + Scale + Rotation)
M_composite = cv2.getRotationMatrix2D(center, 15, 0.85)
M_composite[0, 2] += 20  # Add x translation
M_composite[1, 2] -= 15  # Add y translation
t8 = cv2.warpAffine(img_mri, M_composite, (w, h))
transformations.append(("8. Composite Affine (Scale+Rot+Shift)", t8))

# 9. Non-Linear / Deformable Transformation (Sinusoidal Spatial Warp)
grid_y, grid_x = np.indices((h, w), dtype=np.float32)
# Sinusoidal displacement fields
disp_x = 15.0 * np.sin(2.0 * np.pi * grid_y / 120.0)
disp_y = 15.0 * np.cos(2.0 * np.pi * grid_x / 120.0)
map_x = (grid_x + disp_x).astype(np.float32)
map_y = (grid_y + disp_y).astype(np.float32)
t9 = cv2.remap(img_mri, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
transformations.append(("9. Deformable Sinusoidal Warp", t9))

# 2. Visualize All Transformations in a 3x3 Subplot Grid
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for idx, (title, transformed_img) in enumerate(transformations):
    axes[idx].imshow(transformed_img, cmap='gray')
    axes[idx].set_title(title, fontsize=10)
    axes[idx].axis('off')

plt.suptitle("Geometric Transformations: Affine & Deformable Warping on Brain MRI", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
